In [1]:
import pandas as pd
import numpy as np

# ─── SET THIS EACH YEAR ───────────────────────────────────────────────────────
current_year = 2026   # The EDC lineup we are trying to predict
# ─────────────────────────────────────────────────────────────────────────────

## Data Transformation Explained

To predict if an artist plays in a specific year (e.g. 2026), we restructure the data from
**one row per artist** into **one row per artist-year combination** (the "Long Format").

**Transformation steps:**
1.  **Parse "Years Played":** convert the string `"2022, 2024"` into binary columns `played_2022`, `played_2023`, ...
2.  **Feature engineering:** booking-history features (lags, streak, recency), all popularity metrics
    (log-scaled), and roster/context flags (`is_insomniac`, `has_agency`, `has_residency`, producer ranking).
3.  **Staggered training slices:** instead of a single training block, we build one slice per historical
    target year — features use only what was known *before* year T, target = did they play in year T:
    *   slice 1: 2022–2023 features → did they play 2024?
    *   slice 2: 2022–2024 features → did they play 2025?
    *   (a new slice is added automatically each year as the master dataset grows)
4.  **No manual probability adjustments.** The previous version multiplied model scores by hand-tuned
    factors (+45% Insomniac, ±25% booking-cycle, −30% residency). Backtested against the actual 2026
    lineup those adjustments *hurt* badly (F1 16.3% with them vs 23.5% without, all else equal), so those signals are
    now training features and the model learns their true weight.

In [2]:
# Load datasets
# ---------------------------------------------------------
df_stats = pd.read_csv('../data/main/COMPLETE_edc_artist_and_stats.csv')
df_residency = pd.read_csv(f'../data/extract/{current_year}_vegas_recidency.csv')

# Producer top-100 ranking scraped for the most recent completed year.
# Optional: if the file doesn't exist yet, the feature is simply all zeros.
try:
    df_prod_rank = pd.read_csv(f'../data/extract/{current_year - 1}_producer_rank.csv')
except FileNotFoundError:
    df_prod_rank = pd.DataFrame(columns=['rank', 'artist'])
    print(f"No {current_year - 1}_producer_rank.csv found - producer_rank_score will be 0.")

print(f"Artists in master dataset: {len(df_stats)}")
display(df_stats.head(3))

Artists in master dataset: 973


,artist,followers,streams,playlists,playlist reach,charts,shazams,videos,views,dj supports,followers_growth,streams_growth,total_appearances,years_played,agency
0,1080p,6,5986,1,641,0,17,45,9800,0,0.0,0.0,1,2024,insomniac
1,a shade of black,0,15100,0,11000,7,0,0,0,0,0.0,0.0,1,2024,NaN
2,aaron k,1,2163,0,0,1,4,2,0,0,0.0,0.0,2,"2024, 2025",insomniac


In [ ]:
# Feature Preparation
# ---------------------------------------------------------
import re

HISTORY_YEARS = list(range(2022, current_year))  # e.g. [2022, 2023, 2024, 2025] when current_year=2026

def norm_name(s):
    """Normalize artist names the same way across every data source.
    This is the MATCHING key only - never report it to a human."""
    s = str(s).lower().strip().replace('&', 'and')
    return re.sub(r'\s+', ' ', s)

def parse_years(x):
    if pd.isna(x):
        return []
    return [int(y.strip()) for y in str(x).replace('"', '').split(',') if y.strip().isdigit()]

# Keep the dataset's original spelling for REPORTING before normalizing the
# matching key. Everything downstream joins on `artist` but prints
# `artist_display`, so e.g. 'aly & fila' is matched as 'aly and fila' but
# still shown with the ampersand.
df_stats['artist_display'] = df_stats['artist'].astype(str).str.strip()
df_stats['artist'] = df_stats['artist'].map(norm_name)
df_stats['years_played_list'] = df_stats['years_played'].apply(parse_years)

# Binary played_YYYY columns
for year in HISTORY_YEARS:
    df_stats[f'played_{year}'] = df_stats['years_played_list'].apply(lambda x: 1 if year in x else 0)

# Roster / context flags
df_stats['is_insomniac'] = df_stats['agency'].fillna('').str.lower().str.contains('insomniac').astype(int)
df_stats['has_agency'] = (df_stats['agency'].fillna('') != '').astype(int)

residency_set = set(df_residency['artist'].map(norm_name))
df_stats['has_residency'] = df_stats['artist'].isin(residency_set).astype(int)

# Producer ranking -> score (rank 1 -> 100 points, rank 100 -> 1 point, unranked -> 0)
rank_map = {norm_name(a): 101 - r for r, a in zip(df_prod_rank['rank'], df_prod_rank['artist'])}
df_stats['producer_rank_score'] = df_stats['artist'].map(rank_map).fillna(0)

# Log-scale ALL popularity metrics - they are heavily right-skewed
# (a handful of mega artists dominate the raw counts)
POPULARITY_COLS = ['followers', 'streams', 'playlists', 'playlist reach', 'charts',
                   'shazams', 'videos', 'views', 'dj supports']
LOG_COLS = []
for col in POPULARITY_COLS:
    log_col = f"log_{col.replace(' ', '_')}"
    df_stats[log_col] = np.log1p(df_stats[col].fillna(0))
    LOG_COLS.append(log_col)

print(f"Insomniac artists: {df_stats['is_insomniac'].sum()}")
print(f"Artists with a Vegas residency: {df_stats['has_residency'].sum()}")
print(f"Artists in the producer top-100: {(df_stats['producer_rank_score'] > 0).sum()}")
df_stats.head(3)

In [ ]:
# Build Training Slices (Staggered Time Blocks)
# ---------------------------------------------------------
# One slice per historical target year T: features use only what was known
# BEFORE year T, the target is whether the artist played in year T.
# The earliest usable target is 2024 (it needs at least 2 years of history).

def build_slice(target_year):
    """One row per artist with features known before `target_year`.
    If `target_year` is historical, a `target` column is included."""
    hist = [y for y in HISTORY_YEARS if y < target_year]
    # `artist` is the normalized matching key; `artist_display` carries the
    # original spelling through the pipeline so results can be reported with it.
    out = pd.DataFrame({'artist': df_stats['artist'],
                        'artist_display': df_stats['artist_display']})

    # Lag features (0 if that year predates our data)
    for lag in (1, 2, 3):
        y = target_year - lag
        out[f'played_{lag}_years_ago'] = df_stats[f'played_{y}'] if y in HISTORY_YEARS else 0

    out['total_past_appearances'] = sum(df_stats[f'played_{y}'] for y in hist)
    out['appearance_rate'] = out['total_past_appearances'] / len(hist)

    # Consecutive years played, counting back from target_year - 1 (burnout proxy)
    consecutive = np.zeros(len(df_stats), dtype=int)
    streak_alive = np.ones(len(df_stats), dtype=bool)
    for y in range(target_year - 1, min(hist) - 1, -1):
        streak_alive &= df_stats[f'played_{y}'].to_numpy().astype(bool)
        consecutive += streak_alive
    out['consecutive_years'] = consecutive

    # Years since last appearance (99 = never played)
    last_played = np.full(len(df_stats), np.nan)
    for y in hist:
        last_played = np.where(df_stats[f'played_{y}'] == 1, y, last_played)
    out['years_since_last'] = np.where(np.isnan(last_played), 99, (target_year - 1) - last_played)

    # Popularity + context features (static snapshot)
    out['is_insomniac'] = df_stats['is_insomniac']
    out['has_agency'] = df_stats['has_agency']
    out['has_residency'] = df_stats['has_residency']
    out['producer_rank_score'] = df_stats['producer_rank_score']
    for col in LOG_COLS:
        out[col] = df_stats[col]
    # Growth features appear in the master once >= 2 metrics snapshots exist
    # (notebook 4 computes them from the two most recent cleaned snapshots)
    for col in ('followers_growth', 'streams_growth'):
        if col in df_stats.columns and df_stats[col].fillna(0).abs().sum() > 0:
            out[col] = df_stats[col].fillna(0)

    if target_year in HISTORY_YEARS:
        out['target'] = df_stats[f'played_{target_year}']
    return out

TRAIN_TARGET_YEARS = [y for y in HISTORY_YEARS if y >= 2024]  # [2024, 2025] when current_year=2026
df_train = pd.concat([build_slice(t) for t in TRAIN_TARGET_YEARS], ignore_index=True)

print(f"Training slices: {TRAIN_TARGET_YEARS} -> {df_train.shape[0]} samples "
      f"({df_train['target'].sum()} positives)")
display(df_train.head())

In [ ]:
# Train the Model (Random Forest, tuned)
# ---------------------------------------------------------
from sklearn.ensemble import RandomForestClassifier

# Both name columns are identifiers, not features - keep them out of the matrix.
ID_COLS = ('artist', 'artist_display', 'target')
FEATURES = [c for c in df_train.columns if c not in ID_COLS]

# n_estimators=300 / max_depth=6 were the most stable settings in a
# seed-robustness backtest against the actual 2026 lineup (gradient boosting
# and deeper forests scored worse on this dataset size).
# class_weight='balanced' matters: most artists in the pool don't play in a
# given year, so an unweighted model would just predict "no" for everyone.
model = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42,
                               class_weight='balanced', n_jobs=-1)
model.fit(df_train[FEATURES], df_train['target'])

print(f"Model trained on {len(df_train)} rows x {len(FEATURES)} features.")

# What did the model learn?
importances = (pd.DataFrame({'feature': FEATURES, 'importance': model.feature_importances_})
               .sort_values('importance', ascending=False).reset_index(drop=True))
display(importances)

In [ ]:
# Predict current_year (The Future)
# ---------------------------------------------------------
df_predict = build_slice(current_year)
df_predict['probability'] = model.predict_proba(df_predict[FEATURES])[:, 1].round(5)

# Reported with the original spelling (artist_display), matched on `artist`.
print(f"Predictions generated for {len(df_predict)} artists. Top 10:")
display(df_predict[['artist_display', 'probability']]
        .sort_values('probability', ascending=False)
        .rename(columns={'artist_display': 'artist'})
        .head(10))

In [ ]:
# Select the Predicted Lineup & Save
# ---------------------------------------------------------
TARGET_LINEUP_SIZE = 430  # approximate size of a COMPLETE final lineup (2022-2026 full lineups ran ~370-440 unique artists; the initial announcement is smaller)

final_prediction = df_predict.sort_values('probability', ascending=False).reset_index(drop=True)
if len(final_prediction) >= TARGET_LINEUP_SIZE:
    dynamic_threshold = final_prediction['probability'].iloc[TARGET_LINEUP_SIZE - 1]
else:
    dynamic_threshold = 0.5

final_prediction['result'] = (final_prediction['probability'] >= dynamic_threshold).astype(int)

# Display uses the original spelling; `artist` (normalized) stays in the saved
# CSV so notebook 7 can still match against the scraped lineup.
print(f"--- TOP 50 PREDICTED ARTISTS FOR EDC {current_year} ---")
cols_output = ['artist_display', 'probability', 'result', 'has_residency', 'is_insomniac',
               'played_1_years_ago', 'consecutive_years', 'total_past_appearances']
display(final_prediction[cols_output].rename(columns={'artist_display': 'artist'}).head(50))

output_path = f"../data/result_prediction/edc_{current_year}_prediction.csv"
final_prediction.to_csv(output_path, index=False)
print(f"Saved {final_prediction['result'].sum()} predicted artists "
      f"(of {len(final_prediction)} scored) to {output_path}")